# FACR + RL — Complete Research Pipeline
## Factuality-Aware Contrastive Regularization with RL Early Detection

**Experiments:**
- **A** Full sequence layer sweep — auto L* per dataset
- **B** Early detection sweep — L* re-selected per cutoff
- **C** Cross-dataset transfer
- **D** Contrastive projection head (mixed-cutoff training)
- **E** Before vs after contrastive geometry
- **F** RL adaptive early detection (REINFORCE policy)

> **Before running:** Replace `your_hf_token_here` in Cell 2

> **Run order:** Cell 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11 → 12 → 13 → 14 → 15

### CELL 0 — Setup — directories + cached data

In [ ]:
# ── CELL 0 — Setup ───────────────────────────────────────────────
import os, shutil

os.makedirs('/content/drive/MyDrive/facr_v2', exist_ok=True)
os.makedirs('/content/drive/MyDrive/facr_v2/step1_results', exist_ok=True)
os.makedirs('/content/drive/MyDrive/facr_v2/step2_results', exist_ok=True)

src = '/content/drive/MyDrive/facr_checkpoints/labeled_data.json'
dst = '/content/drive/MyDrive/facr_v2/labeled_data.json'
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copy(src, dst)
    print("labeled_data.json copied from previous session ✓")
elif os.path.exists(dst):
    print("labeled_data.json already exists ✓")
else:
    print("labeled_data.json not found — will regenerate in Cell 4 (~36 min)")

print("Directories ready ✓")


### CELL 1 — Install dependencies

In [ ]:
# ── CELL 1 — Installs ────────────────────────────────────────────
!pip install transformers datasets peft accelerate bitsandbytes tqdm scikit-learn matplotlib -q


### CELL 2 — Imports & model loading

In [ ]:
# ── CELL 2 — Imports & model loading ─────────────────────────────

import os, json, re, string, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.distributions import Bernoulli
from torch.utils.data import Dataset as TorchDataset

from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA

# from google.colab import drive
# drive.mount('/content/drive')
# replace the drive.mount line in Cell 2 with this
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    pass  # already mounted from Cell 0, that's fine

SAVE_DIR  = '/content/drive/MyDrive/facr_v2'
STEP1_DIR = f'{SAVE_DIR}/step1_results'
STEP2_DIR = f'{SAVE_DIR}/step2_results'
os.makedirs(STEP1_DIR, exist_ok=True)
os.makedirs(STEP2_DIR, exist_ok=True)

from huggingface_hub import login
login("<REDACTED>")  # <-- replace

MODEL_ID    = "meta-llama/Llama-3.2-3B-Instruct"
DEVICE      = "cuda"
NUM_LAYERS  = 28
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Device : {DEVICE} | {torch.cuda.get_device_name(0)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    output_hidden_states=True,
)
model.eval()
print(f"Model loaded ✓  |  Layers (incl. embedding): {NUM_LAYERS + 1}")


In [ ]:
# ── Regenerate labeled_data.json ─────────────────────────────────
from datasets import load_dataset
from tqdm import tqdm
import json, os

print("Generating TriviaQA responses (~36 min)...")
print("Progress will show below — do not close the tab\n")

dataset = load_dataset("trivia_qa", "rc.nocontext", split="train[:500]")
labeled_data = []
dist = {'high': 0, 'mid': 0, 'low': 0}

for i, example in enumerate(tqdm(dataset)):
    question = example['question']
    ans      = example['answer']
    prompt   = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )

    # collect all valid gold answer strings
    candidates = []
    if ans.get('value'):
        candidates.append(ans['value'])
    for a in ans.get('aliases', []):
        if isinstance(a, str): candidates.append(a)
    for a in ans.get('normalized_aliases', []):
        if isinstance(a, str): candidates.append(a)

    for _ in range(4):
        resp  = generate_response(question)
        score = max((token_f1(resp, ref)
                     for ref in candidates), default=0.0)

        if   score > 0.5:  dist['high'] += 1
        elif score > 0.1:  dist['mid']  += 1
        else:              dist['low']  += 1

        labeled_data.append({
            'question':         question,
            'response':         resp,
            'factuality_score': float(score),
            'prompt':           prompt
        })

    # checkpoint every 100 questions in case of disconnect
    if (i + 1) % 100 == 0:
        dst = '/content/drive/MyDrive/facr_v2/labeled_data.json'
        with open(dst, 'w') as f:
            json.dump(labeled_data, f)
        print(f"  Checkpoint saved at question {i+1}/500 "
              f"({len(labeled_data)} samples)")

# final save
dst = '/content/drive/MyDrive/facr_v2/labeled_data.json'
with open(dst, 'w') as f:
    json.dump(labeled_data, f)

total = len(labeled_data)
print(f"\nDone!")
print(f"Total samples : {total}")
print(f"Factual (>0.5): {dist['high']} ({100*dist['high']/total:.1f}%)")
print(f"Partial (0.1–0.5): {dist['mid']} ({100*dist['mid']/total:.1f}%)")
print(f"Hallucinated (<0.1): {dist['low']} ({100*dist['low']/total:.1f}%)")
print(f"Saved to: {dst} ✓")

### CELL 3 — Core framework — auto layer selection, no hardcoded layers

In [ ]:
# ── CELL 3 — Core framework functions ────────────────────────────

def normalize(text):
    text = text.lower().strip()
    text = re.sub(r'\b(1\d{2}|\d{2})0s?\b',
                  lambda m: m.group(0).rstrip('s'), text)
    text = re.sub(r'\b(the|a|an)\b', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    return ' '.join(text.split())

def token_f1(pred, ref):
    p_tok = normalize(pred).split()
    r_tok = normalize(ref).split()
    if not p_tok or not r_tok: return 0.0
    common = set(p_tok) & set(r_tok)
    if not common: return 0.0
    p = len(common) / len(p_tok)
    r = len(common) / len(r_tok)
    return 2 * p * r / (p + r)

def generate_response(question, max_new_tokens=40):
    prompt = (
        "Answer with just the name or short phrase, nothing else.\n"
        f"Question: {question}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt",
                       truncation=True, max_length=200).to(DEVICE)
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.8, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return decoded.split('\n')[0].split('.')[0].strip()

def extract_hidden_states(data, layer_idx, cutoff_pct=1.0):
    """
    Extract hidden state at layer_idx, last token position.
    cutoff_pct: fraction of tokens to use (1.0=full, 0.2=first 20%)
    Returns: hiddens (N, hidden_dim), labels (N,)
    """
    hiddens, labels = [], []
    for item in data:
        full_text = item['prompt'] + " " + item['response']
        inputs    = tokenizer(full_text, return_tensors='pt',
                              truncation=True, max_length=256).to(DEVICE)
        if cutoff_pct < 1.0:
            total  = inputs['input_ids'].shape[1]
            cutoff = max(4, int(total * cutoff_pct))
            inputs = {k: v[:, :cutoff] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs, output_hidden_states=True)
        h = out.hidden_states[layer_idx][0, -1, :].float().cpu()
        hiddens.append(h)
        labels.append(1 if item['factuality_score'] > 0.5 else 0)
    return torch.stack(hiddens), torch.tensor(labels)

def quick_auroc(hiddens, labels, cv=3):
    X = hiddens.numpy()
    y = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5
    clf    = LogisticRegression(max_iter=300, C=1.0)
    scores = cross_val_score(clf, X, y, cv=cv, scoring='roc_auc')
    return float(scores.mean())

def robust_auroc(hiddens, labels, cv=10):
    X   = hiddens.numpy()
    y   = labels.numpy()
    if len(np.unique(y)) < 2: return 0.5, 0.0
    skf    = StratifiedKFold(n_splits=cv, shuffle=True,
                             random_state=RANDOM_SEED)
    scores = []
    for tr, te in skf.split(X, y):
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X[tr], y[tr])
        proba = clf.predict_proba(X[te])[:, 1]
        scores.append(roc_auc_score(y[te], proba))
    return float(np.mean(scores)), float(np.std(scores))

def find_and_evaluate(data, cutoff_pct=1.0, verbose=True):
    """
    Auto-finds L* and returns unbiased AUROC.
    3-way split:
      20% layer selection → 60% probe training → 20% evaluation
    No hardcoded layer numbers.
    """
    n         = len(data)
    sel_end   = int(0.2 * n)
    train_end = int(0.8 * n)
    sel_data   = data[:sel_end]
    train_data = data[sel_end:train_end]
    eval_data  = data[train_end:]

    if verbose:
        print(f"  Split → sel:{len(sel_data)} "
              f"| train:{len(train_data)} | eval:{len(eval_data)}")

    # sweep all layers on selection set
    if verbose:
        print(f"  Sweeping {NUM_LAYERS+1} layers at "
              f"cutoff={int(cutoff_pct*100)}% ...")
    layer_aurocs = {}
    for layer_idx in range(NUM_LAYERS + 1):
        h, l = extract_hidden_states(sel_data, layer_idx, cutoff_pct)
        layer_aurocs[layer_idx] = quick_auroc(h, l, cv=3)

    best_layer = max(layer_aurocs, key=layer_aurocs.get)
    if verbose:
        print(f"  L* = layer {best_layer} "
              f"({best_layer/NUM_LAYERS*100:.0f}% depth) | "
              f"sel AUROC = {layer_aurocs[best_layer]:.4f}")

    # train probe at L*
    h_train, l_train = extract_hidden_states(
        train_data, best_layer, cutoff_pct)
    probe = LogisticRegression(max_iter=1000, C=1.0)
    probe.fit(h_train.numpy(), l_train.numpy())

    # evaluate on held-out set
    h_eval, l_eval = extract_hidden_states(
        eval_data, best_layer, cutoff_pct)
    proba       = probe.predict_proba(h_eval.numpy())[:, 1]
    probe_auroc = roc_auc_score(l_eval.numpy(), proba)
    _, probe_std = robust_auroc(h_eval, l_eval, cv=5)

    if verbose:
        print(f"  Final AUROC = {probe_auroc:.4f} ± {probe_std:.4f}")

    return {
        'best_layer':   best_layer,
        'layer_depth':  best_layer / NUM_LAYERS,
        'layer_aurocs': layer_aurocs,
        'probe_auroc':  probe_auroc,
        'probe_std':    probe_std,
        'probe':        probe,
        'h_eval':       h_eval,
        'l_eval':       l_eval,
    }

print("Core framework functions loaded ✓")


In [ ]:
import os, shutil, json

# ── Step 1: find labeled_data.json anywhere on Drive ─────────────
print("Searching for labeled_data.json...")
found_paths = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if 'labeled_data' in f:
            found_paths.append(os.path.join(root, f))

if found_paths:
    for p in found_paths:
        size = os.path.getsize(p)
        print(f"  Found: {p}  ({size/1024:.0f} KB)")
else:
    print("  Not found anywhere on Drive")

# ── Step 2: copy if found ─────────────────────────────────────────
dst = '/content/drive/MyDrive/facr_v2/labeled_data.json'
if found_paths:
    src = found_paths[0]   # use first found
    shutil.copy(src, dst)
    with open(dst) as f:
        d = json.load(f)
    print(f"\nCopied {len(d)} samples to {dst} ✓")
else:
    print("\nlabeled_data.json not found — will regenerate now")
    print("This takes ~36 min. Running generation...")

### CELL 4 — Dataset loaders — TriviaQA, HaluEval (fixed), TruthfulQA, NQ-Open

In [ ]:
# ── CELL 4 — Dataset loaders ──────────────────────────────────────

def load_triviaqa(n=400):
    path = f'{SAVE_DIR}/labeled_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        random.seed(RANDOM_SEED)
        random.shuffle(data)
        print(f"TriviaQA  : {len(data[:n])} samples (from cache)")
        return data[:n]
    raise FileNotFoundError(
        "labeled_data.json not found. "
        "Run facr_colab.py Cell 3 first to generate it."
    )

def load_halueval(n=400):
    """
    Uses correct HaluEval field names:
    question / answer / hallucination (yes=hallucinated, no=factual)
    """
    path = f'{STEP1_DIR}/halueval_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if len(data) > 0:
            print(f"HaluEval  : {len(data[:n])} samples (from cache)")
            return data[:n]
        os.remove(path)

    ds   = load_dataset("pminervini/HaluEval", "qa_samples", split="data")
    data = []
    for ex in list(ds):
        question     = ex.get('question', '').strip()
        answer       = ex.get('answer', '').strip()
        hallucinated = ex.get('hallucination', 'no').strip().lower()
        if not question or not answer:
            continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        factuality = 0.0 if hallucinated == 'yes' else 1.0
        data.append({
            'question':         question,
            'response':         answer,
            'factuality_score': factuality,
            'prompt':           prompt
        })
        if len(data) >= n:
            break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    f_count = sum(1 for d in data[:n] if d['factuality_score'] > 0.5)
    print(f"HaluEval  : {len(data[:n])} samples | "
          f"factual={f_count} | hallu={len(data[:n])-f_count}")
    return data[:n]

def load_truthfulqa(n=400):
    path = f'{STEP1_DIR}/truthfulqa_data.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if len(data) > 0:
            print(f"TruthfulQA: {len(data[:n])} samples (from cache)")
            return data[:n]
        os.remove(path)

    ds   = load_dataset("truthful_qa", "generation", split="validation")
    data = []
    for ex in ds:
        question   = ex.get('question', '').strip()
        best_ans   = ex.get('best_answer', '').strip()
        if not best_ans:
            correct  = ex.get('correct_answers', [])
            best_ans = correct[0] if correct else ''
        wrong_list = ex.get('incorrect_answers', [])
        if not question or not best_ans or not wrong_list:
            continue
        prompt = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        data.append({'question': question, 'response': best_ans,
                     'factuality_score': 1.0, 'prompt': prompt})
        data.append({'question': question, 'response': wrong_list[0],
                     'factuality_score': 0.0, 'prompt': prompt})
        if len(data) >= n:
            break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    print(f"TruthfulQA: {len(data[:n])} samples")
    return data[:n]

def load_nq_open(n=400):
    path = f'{STEP1_DIR}/nq_labeled.json'
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        print(f"NQ-Open   : {len(data[:n])} samples (from cache)")
        return data[:n]

    ds   = load_dataset("nq_open", split="train[:300]")
    data = []
    print("NQ-Open: generating responses (cached after this)...")
    for ex in tqdm(ds):
        question  = ex['question']
        gold_list = ex['answer']
        prompt    = (
            "Answer with just the name or short phrase, nothing else.\n"
            f"Question: {question}\nAnswer:"
        )
        for _ in range(2):
            resp  = generate_response(question)
            score = max(token_f1(resp, ref) for ref in gold_list)
            data.append({'question': question, 'response': resp,
                         'factuality_score': float(score),
                         'prompt': prompt})
        if len(data) >= n:
            break

    random.seed(RANDOM_SEED)
    random.shuffle(data)
    with open(path, 'w') as f:
        json.dump(data[:n], f)
    print(f"NQ-Open   : {len(data[:n])} samples generated and cached")
    return data[:n]

# load all datasets
print("Loading datasets...")
datasets = {
    'TriviaQA':   load_triviaqa(n=400),
    'HaluEval':   load_halueval(n=400),
    'TruthfulQA': load_truthfulqa(n=400),
    'NQ-Open':    load_nq_open(n=400),
}
print("\nDataset summary:")
for name, data in datasets.items():
    f = sum(1 for d in data if d['factuality_score'] > 0.5)
    h = len(data) - f
    print(f"  {name:<12}: {len(data):>4} | "
          f"factual={f} ({100*f/len(data):.0f}%) | "
          f"hallu={h} ({100*h/len(data):.0f}%)")


### CELL 5 — Experiment A — full sequence layer sweep, all datasets

In [ ]:
# ── CELL 5 — EXPERIMENT A: full sequence, all datasets ───────────
# Auto-finds L* per dataset at 100% token cutoff.
# Answers: where does hallucination geometry peak per dataset?

COLORS = {
    'TriviaQA':   'steelblue',
    'HaluEval':   'coral',
    'TruthfulQA': 'mediumseagreen',
    'NQ-Open':    'mediumpurple',
}

print("\n" + "="*60)
print("  EXPERIMENT A — Full sequence, all 4 datasets")
print("="*60)

full_results = {}
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} " + "─"*40)
    result = find_and_evaluate(ds_data, cutoff_pct=1.0, verbose=True)
    full_results[ds_name] = result

save_exp_a = {
    k: {'best_layer': v['best_layer'], 'layer_depth': v['layer_depth'],
        'layer_aurocs': {str(i): a for i, a in v['layer_aurocs'].items()},
        'probe_auroc': v['probe_auroc'], 'probe_std': v['probe_std']}
    for k, v in full_results.items()
}
with open(f'{STEP1_DIR}/exp_a_results.json', 'w') as f:
    json.dump(save_exp_a, f, indent=2)

print("\n── Experiment A Summary ──────────────────────────────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} {'AUROC':>8} {'±Std':>8}")
print("-" * 48)
for ds_name, result in full_results.items():
    print(f"{ds_name:<14} {result['best_layer']:>4} "
          f"{result['layer_depth']*100:>7.0f}%  "
          f"{result['probe_auroc']:>8.4f}  "
          f"±{result['probe_std']:.4f}")

depths = [full_results[n]['layer_depth']*100 for n in full_results]
print(f"\n  Mean L* depth: {np.mean(depths):.1f}% ± {np.std(depths):.1f}%")


### CELL 6 — Experiment B — early detection sweep (L* per cutoff)

In [ ]:
# ── CELL 6 — EXPERIMENT B: early detection sweep ─────────────────
# L* re-selected at each cutoff independently.
# Answers: how early can hallucination be detected?

CUTOFFS = [0.10, 0.20, 0.30, 0.50, 1.00]

print("\n" + "="*60)
print("  EXPERIMENT B — Early detection sweep")
print("  L* re-selected at each cutoff independently")
print("="*60)

early_results = {}
for ds_name in ['TriviaQA', 'HaluEval']:
    early_results[ds_name] = {}
    print(f"\n── {ds_name} " + "─"*40)
    for cutoff in CUTOFFS:
        print(f"\n  Cutoff = {int(cutoff*100)}% tokens:")
        result = find_and_evaluate(
            datasets[ds_name], cutoff_pct=cutoff, verbose=True)
        early_results[ds_name][cutoff] = {
            'best_layer':  result['best_layer'],
            'layer_depth': result['layer_depth'],
            'probe_auroc': result['probe_auroc'],
            'probe_std':   result['probe_std'],
        }

with open(f'{STEP1_DIR}/exp_b_results.json', 'w') as f:
    json.dump(early_results, f, indent=2)

print("\n── Experiment B Summary ──────────────────────────────────")
for ds_name in ['TriviaQA', 'HaluEval']:
    print(f"\n  {ds_name}:")
    print(f"  {'Cutoff':>8} {'L*':>4} {'Depth':>8} "
          f"{'AUROC':>8} {'±Std':>8}")
    print("  " + "-"*44)
    for cutoff in CUTOFFS:
        r = early_results[ds_name][cutoff]
        print(f"  {int(cutoff*100):>6}%  {r['best_layer']:>4}  "
              f"{r['layer_depth']*100:>7.0f}%  "
              f"{r['probe_auroc']:>8.4f}  ±{r['probe_std']:.4f}")


### CELL 7 — Experiment C — cross-dataset transfer

In [ ]:
# ── CELL 7 — EXPERIMENT C: cross-dataset transfer ────────────────
# Train probe on TriviaQA L* → test on all other datasets.
# Answers: does the geometry transfer?

print("\n" + "="*60)
print("  EXPERIMENT C — Cross-dataset transfer")
print("="*60)

tqa_best_layer = full_results['TriviaQA']['best_layer']
print(f"\nTriviaQA L* = {tqa_best_layer} "
      f"({tqa_best_layer/NUM_LAYERS*100:.0f}% depth)")

h_all_tqa, l_all_tqa = extract_hidden_states(
    datasets['TriviaQA'], tqa_best_layer, cutoff_pct=1.0)
transfer_probe = LogisticRegression(max_iter=1000, C=1.0)
transfer_probe.fit(h_all_tqa.numpy(), l_all_tqa.numpy())
print("Transfer probe trained on full TriviaQA ✓")

transfer_results = {}
print(f"\n{'Dataset':<14} {'Transfer':>10} {'Own L*':>8} "
      f"{'Own AUROC':>10} {'Gap':>8}")
print("-" * 55)
for ds_name, ds_data in datasets.items():
    h, l    = extract_hidden_states(ds_data, tqa_best_layer, 1.0)
    proba   = transfer_probe.predict_proba(h.numpy())[:, 1]
    t_auroc = roc_auc_score(l.numpy(), proba)
    own_layer = full_results[ds_name]['best_layer']
    own_auroc = full_results[ds_name]['probe_auroc']
    gap       = own_auroc - t_auroc
    transfer_results[ds_name] = {
        'transfer_auroc': t_auroc, 'own_auroc': own_auroc,
        'own_layer': own_layer, 'transfer_layer': tqa_best_layer,
        'gap': gap,
    }
    note = "← train" if ds_name == 'TriviaQA' else "transfer"
    print(f"{ds_name:<14} {t_auroc:>10.4f} {own_layer:>8} "
          f"{own_auroc:>10.4f} {gap:>+8.4f}  {note}")

with open(f'{STEP1_DIR}/exp_c_results.json', 'w') as f:
    json.dump(transfer_results, f, indent=2)

mean_transfer = np.mean([v['transfer_auroc']
                         for k, v in transfer_results.items()
                         if k != 'TriviaQA'])
print(f"\nMean transfer AUROC (excl. train): {mean_transfer:.4f}")


### CELL 8 — Experiment D — contrastive projection head (mixed-cutoff)

In [ ]:
# ── CELL 8 — EXPERIMENT D: contrastive projection head ───────────
# KEY FIX vs previous version:
#   Train on MIXED cutoffs (20%, 50%, 100%) so the projection
#   head learns geometry that works at early token positions too.
# Llama stays completely frozen — only projection head trains.

class ProjectionHead(nn.Module):
    """
    Lightweight MLP projects hidden states to a geometry-shaped space.
    Only this head trains — base model frozen.
    """
    def __init__(self, input_dim, proj_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

def infonce_loss(z, factuality_scores, temperature=0.07):
    """
    InfoNCE soft contrastive loss. Always positive — no collapse.
    Pulls same-factuality reps together, pushes different ones apart.
    """
    sim     = torch.matmul(z, z.T) / temperature
    f       = factuality_scores.unsqueeze(0) - factuality_scores.unsqueeze(1)
    weights = torch.exp(-f**2 / (2 * 0.5**2))
    weights = weights / weights.sum(dim=1, keepdim=True).clamp(min=1e-8)
    log_probs = F.log_softmax(sim, dim=1)
    return -(weights * log_probs).sum(dim=1).mean()

class ContrastiveDataset(TorchDataset):
    def __init__(self, hiddens, scores):
        self.hiddens = hiddens
        self.scores  = scores
    def __len__(self):
        return len(self.hiddens)
    def __getitem__(self, idx):
        return {'hidden': self.hiddens[idx],
                'factuality_score': self.scores[idx]}

def train_projection_head(train_data, target_layer,
                           proj_dim=256, epochs=10,
                           batch_size=64, lr=1e-3,
                           mixed_cutoffs=None):
    """
    Train projection head with mixed cutoffs.
    mixed_cutoffs: list of cutoff_pct values to sample from per batch.
    If None, uses [0.2, 0.5, 1.0] by default.
    This ensures geometry works at early token positions.
    """
    if mixed_cutoffs is None:
        mixed_cutoffs = [0.2, 0.5, 1.0]

    print(f"  Extracting hidden states (mixed cutoffs: "
          f"{[int(c*100) for c in mixed_cutoffs]}%)...")

    # collect hidden states for all cutoffs
    all_hiddens = []
    all_scores  = []
    for cutoff in mixed_cutoffs:
        h, l = extract_hidden_states(train_data, target_layer, cutoff)
        all_hiddens.append(h)
        all_scores.append(l.float())

    h_combined = torch.cat(all_hiddens, dim=0)
    s_combined = torch.cat(all_scores,  dim=0)
    hidden_dim = h_combined.shape[1]

    # balance classes
    f_idx = (s_combined == 1).nonzero().squeeze()
    h_idx = (s_combined == 0).nonzero().squeeze()
    min_n = min(len(f_idx), len(h_idx))
    bal_idx = torch.cat([f_idx[:min_n], h_idx[:min_n]])
    bal_idx = bal_idx[torch.randperm(len(bal_idx))]
    h_bal   = h_combined[bal_idx]
    s_bal   = s_combined[bal_idx]

    dataset    = ContrastiveDataset(h_bal, s_bal)
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True)

    head      = ProjectionHead(hidden_dim, proj_dim).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs)

    print(f"  Training projection head "
          f"({sum(p.numel() for p in head.parameters()):,} params) "
          f"for {epochs} epochs on mixed cutoffs...")

    head.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch in dataloader:
            h = batch['hidden'].to(DEVICE)
            s = batch['factuality_score'].to(DEVICE)
            optimizer.zero_grad()
            z    = head(h)
            loss = infonce_loss(z, s, temperature=0.07)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:2d}/{epochs} | "
                  f"loss={epoch_loss/len(dataloader):.4f}")
        scheduler.step()

    head.eval()
    print("  Projection head trained ✓")
    return head

def evaluate_with_head(head, data, target_layer, cutoff_pct=1.0):
    h, l = extract_hidden_states(data, target_layer, cutoff_pct)
    with torch.no_grad():
        z = head(h.to(DEVICE)).cpu()
    mean, std = robust_auroc(z, l, cv=10)
    return z, l, mean, std

print("\n" + "="*60)
print("  EXPERIMENT D — Contrastive projection (mixed-cutoff)")
print("  Frozen model + projection head at TriviaQA L*")
print("="*60)

tqa_layer  = full_results['TriviaQA']['best_layer']
tqa_train  = datasets['TriviaQA'][:int(0.8 * len(datasets['TriviaQA']))]

print(f"\nTarget L* = {tqa_layer} "
      f"({tqa_layer/NUM_LAYERS*100:.0f}% depth) [auto-selected]")

proj_head = train_projection_head(
    tqa_train, tqa_layer,
    proj_dim=256, epochs=10,
    batch_size=64, lr=1e-3,
    mixed_cutoffs=[0.2, 0.5, 1.0]
)
torch.save(proj_head.state_dict(),
           f'{STEP2_DIR}/proj_head_layer{tqa_layer}_mixed.pt')
print(f"Projection head saved ✓")


### CELL 9 — Experiment E — before vs after contrastive geometry

In [ ]:
# ── CELL 9 — EXPERIMENT E: before vs after geometry ──────────────

print("\n" + "="*60)
print("  EXPERIMENT E — Before vs after contrastive geometry")
print("="*60)

contrastive_results = {}
print("\nEvaluating on all datasets + all cutoffs...")
for ds_name, ds_data in datasets.items():
    print(f"\n── {ds_name} ──────────────────────────────────────────")
    ds_eval = ds_data[int(0.8*len(ds_data)):]

    h_raw, l_raw      = extract_hidden_states(ds_eval, tqa_layer, 1.0)
    raw_mean, raw_std = robust_auroc(h_raw, l_raw, cv=10)

    z_proj, l_proj, proj_mean, proj_std = evaluate_with_head(
        proj_head, ds_eval, tqa_layer, 1.0)

    delta = proj_mean - raw_mean
    contrastive_results[ds_name] = {
        'raw_auroc':  raw_mean, 'raw_std':    raw_std,
        'proj_auroc': proj_mean, 'proj_std':  proj_std,
        'delta':      delta,
        'h_raw':      h_raw, 'z_proj': z_proj, 'labels': l_raw,
    }
    print(f"  Raw hidden : AUROC = {raw_mean:.4f} ± {raw_std:.4f}")
    print(f"  Projected  : AUROC = {proj_mean:.4f} ± {proj_std:.4f}")
    print(f"  Delta      : {delta:+.4f}")

# early detection: raw vs projected (with mixed-cutoff training fix)
print("\n── Early detection raw vs projected (TriviaQA) ──────────")
print(f"{'Cutoff':>8} {'Raw':>10} {'Projected':>12} {'Delta':>8}")
print("-" * 44)

early_contrastive = {}
for cutoff in CUTOFFS:
    ds_eval           = datasets['TriviaQA'][int(
        0.8*len(datasets['TriviaQA'])):]
    h_raw, l_raw      = extract_hidden_states(ds_eval, tqa_layer, cutoff)
    raw_mean, _       = robust_auroc(h_raw, l_raw, cv=5)
    z_proj, l_proj, proj_mean, _ = evaluate_with_head(
        proj_head, ds_eval, tqa_layer, cutoff)
    delta = proj_mean - raw_mean
    early_contrastive[cutoff] = {
        'raw_auroc': raw_mean, 'proj_auroc': proj_mean, 'delta': delta}
    print(f"{int(cutoff*100):>7}%  {raw_mean:>10.4f}  "
          f"{proj_mean:>12.4f}  {delta:>+8.4f}")


### CELL 10 — Experiment F — RL policy definitions (REINFORCE)

In [ ]:
# ── CELL 10 — EXPERIMENT F: RL policy definitions ────────────────
# REINFORCE policy that adaptively decides WHEN to flag hallucination.
# State:  hidden state at L* at current token
# Action: 0=continue, 1=flag as hallucination
# Reward: +1 correct early flag, -1 wrong flag, small per-token signals

class HallucinationPolicy(nn.Module):
    """
    2-layer MLP policy on top of frozen hidden states.
    Input : hidden state at L* (3072-dim for Llama-3.2-3B)
    Output: probability of flagging as hallucination
    ~50K params — deliberately tiny
    """
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def act(self, state):
        with torch.no_grad():
            prob = self.forward(state)
        dist     = Bernoulli(prob)
        action   = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob, prob.item()

def compute_reward(action, true_label, token_position,
                   total_tokens, alpha=0.3):
    """
    Reward shaping with early detection bonus.
    alpha controls how much we reward early flagging.
    """
    pct_used    = token_position / max(total_tokens, 1)
    early_bonus = alpha * (1.0 - pct_used)

    if action == 1:            # flagged as hallucination
        if true_label == 0:    # correct — it WAS hallucinated
            return 1.0 + early_bonus
        else:                  # wrong — it was factual
            return -1.0
    else:                      # continued generating
        if true_label == 0:    # should have flagged
            return -0.1        # small per-token penalty
        else:                  # correct to continue
            return 0.05        # small per-token reward

def run_episode(policy, item, target_layer, device='cuda'):
    """
    Token-by-token rollout for one sample.
    Policy sees hidden state at each position and decides: flag or continue.
    Episode ends when policy flags or sequence ends.
    """
    full_text  = item['prompt'] + " " + item['response']
    true_label = 1 if item['factuality_score'] > 0.5 else 0
    inputs     = tokenizer(full_text, return_tensors='pt',
                           truncation=True, max_length=256).to(device)
    total_tokens = inputs['input_ids'].shape[1]
    min_tokens   = max(4, int(0.05 * total_tokens))

    log_probs     = []
    rewards       = []
    flag_position = None
    final_action  = 0

    for t in range(min_tokens, total_tokens):
        current_inputs = {k: v[:, :t] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**current_inputs, output_hidden_states=True)
        h = out.hidden_states[target_layer][0, -1, :].float()

        action, log_prob, prob = policy.act(h.unsqueeze(0).to(device))
        log_probs.append(log_prob)
        rewards.append(
            compute_reward(action, true_label, t, total_tokens, alpha=0.3))

        if action == 1:
            flag_position = t
            final_action  = 1
            break

    correct = (final_action == 1 and true_label == 0) or \
              (final_action == 0 and true_label == 1)

    return log_probs, rewards, flag_position, correct, total_tokens

def train_rl_policy(train_data, target_layer,
                    episodes=500, lr=1e-3,
                    gamma=0.99, device='cuda'):
    """
    REINFORCE training.
    For each episode:
      G_t = discounted future rewards from step t
      loss = -sum(log_prob(a_t) * G_t)
    """
    sample_inputs = tokenizer(
        train_data[0]['prompt'], return_tensors='pt',
        truncation=True, max_length=50).to(device)
    with torch.no_grad():
        sample_out = model(**sample_inputs, output_hidden_states=True)
    hidden_dim = sample_out.hidden_states[target_layer].shape[-1]

    policy    = HallucinationPolicy(hidden_dim, hidden_dim=128).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=episodes)

    print(f"  Policy params : "
          f"{sum(p.numel() for p in policy.parameters()):,}")
    print(f"  Hidden dim    : {hidden_dim}")
    print(f"  Target layer  : {target_layer} "
          f"({target_layer/NUM_LAYERS*100:.0f}% depth)")
    print(f"  Training for {episodes} episodes...")

    ep_rewards  = []
    ep_accuracy = []
    best_acc    = 0

    for ep in range(episodes):
        item = random.choice(train_data)
        log_probs, rewards, flag_pos, correct, total_tokens = \
            run_episode(policy, item, target_layer, device)

        if len(log_probs) == 0:
            continue

        # discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)

        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        log_probs_t = torch.stack(log_probs).to(device)
        loss        = -(log_probs_t * returns).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        ep_rewards.append(sum(rewards))
        ep_accuracy.append(float(correct))

        if (ep + 1) % 50 == 0:
            recent_acc = np.mean(ep_accuracy[-50:])
            recent_rew = np.mean(ep_rewards[-50:])
            flag_info  = (f"flagged at {flag_pos}/{total_tokens} "
                         f"({100*flag_pos/total_tokens:.0f}%)"
                         if flag_pos else "did not flag")
            print(f"  Ep {ep+1:4d} | acc={recent_acc:.3f} | "
                  f"reward={recent_rew:.3f} | {flag_info}")

            if recent_acc > best_acc:
                best_acc = recent_acc
                torch.save(policy.state_dict(),
                           f'{STEP2_DIR}/rl_policy_best.pt')

    print(f"\n  Best accuracy: {best_acc:.4f}")
    return policy, ep_rewards, ep_accuracy

print("RL framework definitions loaded ✓")
print(f"Will train at L* = {tqa_layer} "
      f"(auto-selected from TriviaQA Experiment A)")


### CELL 11 — Experiment F — RL training on TriviaQA

In [ ]:
# ── Fix: replace the act() method and run_episode() ──────────────

class HallucinationPolicy(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def act(self, state):
        # ── KEY FIX: prob computed with grad, action sampled without ──
        prob   = self.forward(state)           # keeps grad_fn
        dist   = Bernoulli(prob)
        with torch.no_grad():
            action = dist.sample()             # sample without grad
        log_prob = dist.log_prob(action)       # log_prob KEEPS grad_fn
        return action.item(), log_prob, prob.item()

def run_episode(policy, item, target_layer, device='cuda'):
    full_text  = item['prompt'] + " " + item['response']
    true_label = 1 if item['factuality_score'] > 0.5 else 0
    inputs     = tokenizer(full_text, return_tensors='pt',
                           truncation=True, max_length=256).to(device)
    total_tokens = inputs['input_ids'].shape[1]
    min_tokens   = max(4, int(0.05 * total_tokens))

    log_probs     = []
    rewards       = []
    flag_position = None
    final_action  = 0

    for t in range(min_tokens, total_tokens):
        current_inputs = {k: v[:, :t] for k, v in inputs.items()}

        # ── extract hidden state with no_grad (Llama frozen) ─────────
        with torch.no_grad():
            out = model(**current_inputs, output_hidden_states=True)
        h = out.hidden_states[target_layer][0, -1, :].float()

        # ── policy decision — grad flows through policy only ─────────
        action, log_prob, prob = policy.act(
            h.unsqueeze(0).to(device)
        )

        log_probs.append(log_prob)
        rewards.append(
            compute_reward(action, true_label, t,
                           total_tokens, alpha=0.3))

        if action == 1:
            flag_position = t
            final_action  = 1
            break

    correct = (final_action == 1 and true_label == 0) or \
              (final_action == 0 and true_label == 1)

    return log_probs, rewards, flag_position, correct, total_tokens

print("Fixed HallucinationPolicy and run_episode ✓")

In [ ]:
# ── CELL 11 — EXPERIMENT F: RL training ──────────────────────────

print("\n" + "="*60)
print("  EXPERIMENT F — RL adaptive early detection policy")
print("  REINFORCE on frozen Llama-3.2-3B hidden states")
print("="*60)

tqa_train_rl = datasets['TriviaQA'][:int(
    0.8 * len(datasets['TriviaQA']))]

print(f"\nTraining on TriviaQA | "
      f"L* = {tqa_layer} | "
      f"samples = {len(tqa_train_rl)}")

rl_policy, ep_rewards, ep_accuracy = train_rl_policy(
    tqa_train_rl,
    target_layer=tqa_layer,
    episodes=500,
    lr=1e-3,
    gamma=0.99,
    device=DEVICE
)


### CELL 12 — Experiment F — RL evaluation vs fixed cutoff baselines

In [ ]:
# ── CELL 12 — EXPERIMENT F: RL evaluation ────────────────────────

def evaluate_rl_policy(policy, eval_data, target_layer, device='cuda'):
    """
    Evaluate trained RL policy.
    Metrics: AUROC, accuracy, avg token position of flag.
    """
    policy.eval()
    all_probs    = []
    all_labels   = []
    flag_pcts    = []
    correct_list = []

    for item in tqdm(eval_data, desc="Evaluating RL"):
        full_text    = item['prompt'] + " " + item['response']
        true_label   = 1 if item['factuality_score'] > 0.5 else 0
        inputs       = tokenizer(full_text, return_tensors='pt',
                                 truncation=True, max_length=256).to(device)
        total_tokens = inputs['input_ids'].shape[1]
        min_tokens   = max(4, int(0.05 * total_tokens))

        max_flag_prob = 0.0
        flag_position = None

        for t in range(min_tokens, total_tokens):
            current_inputs = {k: v[:, :t] for k, v in inputs.items()}
            with torch.no_grad():
                out = model(**current_inputs, output_hidden_states=True)
            h    = out.hidden_states[target_layer][0, -1, :].float()
            prob = policy(h.unsqueeze(0).to(device)).item()

            if prob > max_flag_prob:
                max_flag_prob = prob
            if prob > 0.5 and flag_position is None:
                flag_position = t

        all_probs.append(max_flag_prob)
        all_labels.append(true_label)
        predicted_hallu = 1 if max_flag_prob > 0.5 else 0
        correct_list.append(
            1 if predicted_hallu == (1 - true_label) else 0)
        if flag_position is not None:
            flag_pcts.append(flag_position / total_tokens)

    accuracy     = np.mean(correct_list)
    auroc        = roc_auc_score([1-l for l in all_labels], all_probs)
    avg_flag_pct = np.mean(flag_pcts) if flag_pcts else 1.0
    flag_rate    = len(flag_pcts) / len(eval_data)

    return {'accuracy': accuracy, 'auroc': auroc,
            'avg_flag_pct': avg_flag_pct, 'flag_rate': flag_rate,
            'n_eval': len(eval_data)}

tqa_eval_rl = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]
print(f"Evaluating on {len(tqa_eval_rl)} held-out TriviaQA samples...")

rl_results = evaluate_rl_policy(
    rl_policy, tqa_eval_rl, tqa_layer, device=DEVICE)

print(f"\n── RL Policy Results (TriviaQA) ──────────────────────")
print(f"  Accuracy    : {rl_results['accuracy']:.4f}")
print(f"  AUROC       : {rl_results['auroc']:.4f}")
print(f"  Avg flag at : {rl_results['avg_flag_pct']*100:.1f}% of tokens")
print(f"  Flag rate   : {rl_results['flag_rate']*100:.1f}% flagged")

print(f"\n── RL vs fixed cutoff comparison ─────────────────────")
print(f"  {'Method':<35} {'AUROC':>8} {'Tokens':>8}")
print(f"  {'-'*54}")
for cutoff in CUTOFFS:
    auroc = early_results['TriviaQA'][cutoff]['probe_auroc']
    print(f"  {'Fixed probe @ '+str(int(cutoff*100))+'% tokens':<35} "
          f"{auroc:>8.4f} {int(cutoff*100):>7}%")
print(f"  {'RL adaptive policy':<35} "
      f"{rl_results['auroc']:>8.4f} "
      f"{rl_results['avg_flag_pct']*100:>7.1f}%  ← learned")


### CELL 13 — Experiment F — cross-dataset RL evaluation

In [ ]:
# ── CELL 13 — EXPERIMENT F: cross-dataset RL ─────────────────────

print(f"\n── Cross-dataset RL evaluation ───────────────────────")
print(f"  Policy trained on TriviaQA L*={tqa_layer}\n")
print(f"  {'Dataset':<14} {'AUROC':>8} {'Accuracy':>10} "
      f"{'Flag at':>10} {'Flag rate':>10}")
print(f"  {'-'*56}")

cross_dataset_rl = {}
for ds_name, ds_data in datasets.items():
    ds_eval = ds_data[int(0.8*len(ds_data)):]
    res     = evaluate_rl_policy(
        rl_policy, ds_eval, tqa_layer, device=DEVICE)
    cross_dataset_rl[ds_name] = res
    print(f"  {ds_name:<14} {res['auroc']:>8.4f} "
          f"{res['accuracy']:>10.4f} "
          f"{res['avg_flag_pct']*100:>9.1f}% "
          f"{res['flag_rate']*100:>9.1f}%")

with open(f'{STEP2_DIR}/exp_f_results.json', 'w') as f:
    json.dump({k: {kk: vv for kk, vv in v.items()}
               for k, v in cross_dataset_rl.items()}, f, indent=2)


### CELL 14 — All figures — 6 publication-ready plots

In [ ]:
# ── CELL 14 — All figures ─────────────────────────────────────────

# ── Figure 1: Layer AUROC curves per dataset ──────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4), sharey=True)
for ax, (ds_name, result) in zip(axes, full_results.items()):
    color  = COLORS[ds_name]
    layers = sorted(result['layer_aurocs'].keys())
    aurocs = [result['layer_aurocs'][l] for l in layers]
    best   = result['best_layer']
    ax.plot(layers, aurocs, '-', color=color, linewidth=1.5, alpha=0.8)
    ax.scatter(layers, aurocs, color=color, s=20, zorder=4)
    ax.axvline(best, color='black', linestyle='--',
               linewidth=1.2, alpha=0.7)
    ax.scatter([best], [result['layer_aurocs'][best]],
               color='black', s=100, zorder=5,
               label=f"L*={best} ({best/NUM_LAYERS*100:.0f}%)")
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(f"{ds_name}\nAUROC={result['probe_auroc']:.3f}",
                 fontsize=10)
    ax.set_xlabel('Layer', fontsize=9)
    ax.set_ylim(0.4, 1.05)
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.2)
    if ax == axes[0]:
        ax.set_ylabel('AUROC (probe)', fontsize=10)
plt.suptitle('Auto L* selection across 4 datasets', fontsize=12)
plt.tight_layout()
plt.savefig(f'{STEP1_DIR}/fig1_layer_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1 saved ✓")

# ── Figure 2: Early detection raw vs projected ────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
cutoffs   = sorted(early_contrastive.keys())
raw_vals  = [early_contrastive[c]['raw_auroc']  for c in cutoffs]
proj_vals = [early_contrastive[c]['proj_auroc'] for c in cutoffs]
x_labels  = [f"{int(c*100)}%" for c in cutoffs]
xi        = range(len(cutoffs))
ax.plot(xi, raw_vals,  'o-', color='steelblue', linewidth=2,
        markersize=8, label='Raw hidden states')
ax.plot(xi, proj_vals, 's-', color='coral',     linewidth=2,
        markersize=8, label='Contrastive projection (mixed-cutoff)')
ax.fill_between(xi,
    [max(0, p-r) for p, r in zip(proj_vals, raw_vals)],
    alpha=0.1, color='coral', label='Improvement')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.axhline(0.7, color='green', linestyle='--', alpha=0.5,
           label='0.7 threshold')
ax.set_xticks(xi)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Tokens generated', fontsize=11)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_title(f'Early detection: raw vs contrastive projection\n'
             f'Layer {tqa_layer} — TriviaQA', fontsize=11)
ax.set_ylim(0.4, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig2_early_detection_contrast.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 2 saved ✓")

# ── Figure 3: RL training curves ─────────────────────────────────
def smooth(vals, window=20):
    return [np.mean(vals[max(0,i-window):i+1])
            for i in range(len(vals))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot(smooth(ep_accuracy, 20), color='steelblue',
        linewidth=1.5, label='Accuracy (smoothed)')
ax.plot(ep_accuracy, color='steelblue', alpha=0.2, linewidth=0.5)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Chance')
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Accuracy', fontsize=11)
ax.set_title('RL training — accuracy', fontsize=11)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(smooth(ep_rewards, 20), color='coral',
        linewidth=1.5, label='Reward (smoothed)')
ax.plot(ep_rewards, color='coral', alpha=0.2, linewidth=0.5)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Reward', fontsize=11)
ax.set_title('RL training — reward', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle(f'REINFORCE training on TriviaQA | '
             f'L*={tqa_layer} | Llama-3.2-3B frozen',
             fontsize=12)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig3_rl_training.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 3 saved ✓")

# ── Figure 4: RL vs fixed cutoff comparison ───────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
methods  = [f"Probe {int(c*100)}%" for c in CUTOFFS] + ['RL (adaptive)']
aurocs_f = [early_results['TriviaQA'][c]['probe_auroc']
            for c in CUTOFFS] + [rl_results['auroc']]
token_pcts = [int(c*100) for c in CUTOFFS] + \
             [int(rl_results['avg_flag_pct']*100)]
bar_colors = ['steelblue']*len(CUTOFFS) + ['coral']
x = np.arange(len(methods))
bars = ax.bar(x, aurocs_f, color=bar_colors,
              edgecolor='white', width=0.6)
for bar, auroc, tok in zip(bars, aurocs_f, token_pcts):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005,
            f'{auroc:.3f}\n({tok}% tokens)',
            ha='center', va='bottom', fontsize=8)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(methods, rotation=15, fontsize=9)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_ylim(0.4, 1.05)
ax.set_title('RL adaptive policy vs fixed cutoff probes\n'
             '(TriviaQA — orange = RL, blue = fixed probe)',
             fontsize=11)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig4_rl_vs_probe.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 4 saved ✓")

# ── Figure 5: PCA before vs after ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
res = contrastive_results['TriviaQA']
for ax, hiddens, title in [
    (axes[0], res['h_raw'],  f'Raw hidden states (layer {tqa_layer})'),
    (axes[1], res['z_proj'], 'After contrastive projection')
]:
    y       = res['labels'].numpy()
    pca     = PCA(n_components=2)
    reduced = pca.fit_transform(hiddens.numpy())
    f_mask  = y == 1
    ax.scatter(reduced[f_mask,  0], reduced[f_mask,  1],
               c='steelblue', alpha=0.6, s=18, label='Factual')
    ax.scatter(reduced[~f_mask, 0], reduced[~f_mask, 1],
               c='coral',     alpha=0.6, s=18, label='Hallucinated')
    h_t = torch.tensor(hiddens.numpy())
    fc  = h_t[f_mask].mean(0)
    hc  = h_t[~f_mask].mean(0)
    sim = F.cosine_similarity(fc.unsqueeze(0), hc.unsqueeze(0)).item()
    ax.set_title(f'{title}\nCentroid sim={sim:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle(f'Activation geometry before vs after contrastive '
             f'projection\nTriviaQA — layer {tqa_layer}',
             fontsize=12)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig5_pca_before_after.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 5 saved ✓")

# ── Figure 6: Cross-dataset RL ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ds_list    = list(cross_dataset_rl.keys())
rl_aurocs  = [cross_dataset_rl[d]['auroc']     for d in ds_list]
own_aurocs = [full_results[d]['probe_auroc']   for d in ds_list]
x = np.arange(len(ds_list))
w = 0.35
ax.bar(x-w/2, own_aurocs, w, label='Probe (own L*)',
       color='steelblue', alpha=0.85)
ax.bar(x+w/2, rl_aurocs,  w, label='RL policy (TriviaQA L*)',
       color='coral',     alpha=0.85)
for i, (o, r) in enumerate(zip(own_aurocs, rl_aurocs)):
    ax.text(i-w/2, o+0.005, f'{o:.3f}',
            ha='center', va='bottom', fontsize=8)
    ax.text(i+w/2, r+0.005, f'{r:.3f}',
            ha='center', va='bottom', fontsize=8)
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xticks(x)
ax.set_xticklabels(ds_list, fontsize=9)
ax.set_ylabel('AUROC', fontsize=11)
ax.set_ylim(0.4, 1.1)
ax.set_title('Cross-dataset: probe vs RL policy', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{STEP2_DIR}/fig6_cross_dataset_rl.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Figure 6 saved ✓")


### CELL 15 — Final results table — all experiments, paper ready

In [ ]:
# ── CELL 15 — Final results table ────────────────────────────────

print("\n" + "="*60)
print("  FINAL RESULTS TABLE — PAPER READY")
print("="*60)
print(f"\nModel  : Llama-3.2-3B-Instruct (frozen throughout)")
print(f"Method : Auto L* + contrastive projection + RL policy")

print(f"\n── Table 1: Layer selection (Experiment A) ───────────────")
print(f"{'Dataset':<14} {'L*':>4} {'Depth':>8} {'AUROC':>8} {'±Std':>8}")
print("-" * 50)
for ds_name in datasets.keys():
    r = full_results[ds_name]
    print(f"{ds_name:<14} {r['best_layer']:>4} "
          f"{r['layer_depth']*100:>7.0f}%  "
          f"{r['probe_auroc']:>8.4f}  ±{r['probe_std']:.4f}")
depths_list = [full_results[n]['layer_depth']*100 for n in datasets]
print(f"\n  Mean L* depth: {np.mean(depths_list):.1f}% "
      f"± {np.std(depths_list):.1f}%")

print(f"\n── Table 2: Early detection (Experiment B — TriviaQA) ────")
print(f"{'Cutoff':>8} {'Raw AUROC':>12} {'Proj AUROC':>12} {'Delta':>8}")
print("-" * 46)
for cutoff in CUTOFFS:
    r = early_contrastive[cutoff]
    print(f"{int(cutoff*100):>7}%  {r['raw_auroc']:>12.4f}  "
          f"{r['proj_auroc']:>12.4f}  {r['delta']:>+8.4f}")

print(f"\n── Table 3: Cross-dataset transfer (Experiment C) ────────")
print(f"{'Dataset':<14} {'Own L*':>6} {'Own AUROC':>10} "
      f"{'Transfer':>10} {'Gap':>8}")
print("-" * 52)
for ds_name in datasets.keys():
    t = transfer_results[ds_name]
    print(f"{ds_name:<14} {t['own_layer']:>6}  "
          f"{t['own_auroc']:>10.4f}  "
          f"{t['transfer_auroc']:>10.4f}  {t['gap']:>+8.4f}")

print(f"\n── Table 4: Contrastive projection (Experiment D+E) ──────")
print(f"{'Dataset':<14} {'Raw':>8} {'Projected':>10} {'Delta':>8}")
print("-" * 44)
for ds_name, r in contrastive_results.items():
    print(f"{ds_name:<14} {r['raw_auroc']:>8.4f}  "
          f"{r['proj_auroc']:>10.4f}  {r['delta']:>+8.4f}")

print(f"\n── Table 5: RL adaptive policy (Experiment F) ────────────")
print(f"{'Method':<35} {'AUROC':>8} {'Tokens':>10}")
print("-" * 56)
for cutoff in CUTOFFS:
    auroc = early_results['TriviaQA'][cutoff]['probe_auroc']
    print(f"{'Fixed probe @ '+str(int(cutoff*100))+'% tokens':<35} "
          f"{auroc:>8.4f} {int(cutoff*100):>9}%")
print(f"{'RL adaptive policy':<35} "
      f"{rl_results['auroc']:>8.4f} "
      f"{rl_results['avg_flag_pct']*100:>9.1f}%  ← learned")

print(f"\n── Cross-dataset RL ───────────────────────────────────────")
print(f"{'Dataset':<14} {'AUROC':>8} {'Avg flag at':>12} "
      f"{'Flag rate':>10}")
print("-" * 48)
for ds_name, res in cross_dataset_rl.items():
    print(f"{ds_name:<14} {res['auroc']:>8.4f} "
          f"{res['avg_flag_pct']*100:>11.1f}%  "
          f"{res['flag_rate']*100:>9.1f}%")

print(f"\n── Key findings ──────────────────────────────────────────")
all_layers = [full_results[n]['best_layer'] for n in datasets]
print(f"1. L* identified automatically — layers found: {all_layers}")
print(f"   Mean depth: {np.mean(depths_list):.1f}% ± "
      f"{np.std(depths_list):.1f}% (task-dependent)")
xfer_vals = [v['transfer_auroc'] for k, v in transfer_results.items()
             if k != 'TriviaQA']
print(f"2. Transfer AUROC: {np.mean(xfer_vals):.4f} mean")
c_deltas = [r['delta'] for r in contrastive_results.values()]
print(f"3. Contrastive projection mean delta: "
      f"{np.mean(c_deltas):+.4f}")
print(f"4. RL policy AUROC: {rl_results['auroc']:.4f} at "
      f"{rl_results['avg_flag_pct']*100:.1f}% tokens (adaptive)")

print(f"\nFigures saved:")
for fig_path in [
    f'{STEP1_DIR}/fig1_layer_curves.png',
    f'{STEP2_DIR}/fig2_early_detection_contrast.png',
    f'{STEP2_DIR}/fig3_rl_training.png',
    f'{STEP2_DIR}/fig4_rl_vs_probe.png',
    f'{STEP2_DIR}/fig5_pca_before_after.png',
    f'{STEP2_DIR}/fig6_cross_dataset_rl.png',
]:
    print(f"  {fig_path}")
print("="*60)


Rechecking the RL results


In [ ]:
# ── CELL 10 — EXPERIMENT F: RL policy definitions ────────────────

from torch.distributions import Bernoulli

class HallucinationPolicy(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

    def act(self, state):
        prob     = self.forward(state)          # keeps grad_fn
        dist     = Bernoulli(prob)
        with torch.no_grad():
            action = dist.sample()
        log_prob = dist.log_prob(action)        # keeps grad_fn
        return action.item(), log_prob, prob.item()

def compute_reward(action, true_label, token_position,
                   total_tokens, alpha=0.3):
    """
    Rebalanced rewards to prevent always-flag collapse.
    Wrong flags penalized harder, correct non-flags rewarded more.
    """
    pct_used    = token_position / max(total_tokens, 1)
    early_bonus = alpha * (1.0 - pct_used)

    if action == 1:             # flagged as hallucination
        if true_label == 0:     # correct — reward + early bonus
            return 1.0 + early_bonus
        else:                   # wrong — stronger penalty than before
            return -2.0
    else:                       # continued generating
        if true_label == 0:     # should have flagged — penalty
            return -0.2
        else:                   # correct to continue — reward
            return 0.1

def run_episode(policy, item, target_layer, device='cuda'):
    full_text  = item['prompt'] + " " + item['response']
    true_label = 1 if item['factuality_score'] > 0.5 else 0
    inputs     = tokenizer(full_text, return_tensors='pt',
                           truncation=True, max_length=256).to(device)
    total_tokens = inputs['input_ids'].shape[1]
    min_tokens   = max(4, int(0.05 * total_tokens))

    log_probs     = []
    rewards       = []
    probs_list    = []    # track for entropy
    flag_position = None
    final_action  = 0

    for t in range(min_tokens, total_tokens):
        current_inputs = {k: v[:, :t] for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**current_inputs, output_hidden_states=True)
        h = out.hidden_states[target_layer][0, -1, :].float()

        action, log_prob, prob = policy.act(h.unsqueeze(0).to(device))
        log_probs.append(log_prob)
        probs_list.append(prob)
        rewards.append(
            compute_reward(action, true_label, t,
                           total_tokens, alpha=0.3))
        if action == 1:
            flag_position = t
            final_action  = 1
            break

    correct = (final_action == 1 and true_label == 0) or \
              (final_action == 0 and true_label == 1)

    return log_probs, rewards, flag_position, correct, total_tokens

def train_rl_policy(train_data, target_layer,
                    episodes=2000, lr=5e-4,
                    gamma=0.99, device='cuda'):
    """
    REINFORCE with entropy regularization.
    Entropy bonus prevents policy from collapsing to always-flag.
    """
    sample_inputs = tokenizer(
        train_data[0]['prompt'], return_tensors='pt',
        truncation=True, max_length=50).to(device)
    with torch.no_grad():
        sample_out = model(**sample_inputs, output_hidden_states=True)
    hidden_dim = sample_out.hidden_states[target_layer].shape[-1]

    policy    = HallucinationPolicy(hidden_dim, hidden_dim=128).to(device)
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=episodes)

    print(f"  Policy params : "
          f"{sum(p.numel() for p in policy.parameters()):,}")
    print(f"  Hidden dim    : {hidden_dim}")
    print(f"  Target layer  : {target_layer} "
          f"({target_layer/NUM_LAYERS*100:.0f}% depth)")
    print(f"  Episodes      : {episodes}")
    print(f"  Entropy reg   : 0.01 (prevents always-flag collapse)")

    ep_rewards  = []
    ep_accuracy = []
    flag_rates  = []
    best_acc    = 0

    for ep in range(episodes):
        item = random.choice(train_data)
        log_probs, rewards, flag_pos, correct, total_tokens = \
            run_episode(policy, item, target_layer, device)

        if len(log_probs) == 0:
            continue

        # discounted returns
        returns = []
        G = 0
        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)
        returns = torch.tensor(returns,
                               dtype=torch.float32).to(device)

        if returns.std() > 1e-8:
            returns = (returns - returns.mean()) / \
                      (returns.std() + 1e-8)

        log_probs_t = torch.stack(log_probs).to(device)

        # REINFORCE loss + entropy regularization
        # entropy bonus discourages always-flag / always-continue
        policy_loss = -(log_probs_t * returns).mean()
        entropy_reg = -0.01 * torch.stack([
            Bernoulli(torch.tensor(p)).entropy()
            for p in [abs(lp.exp().item())
                      for lp in log_probs]
        ]).mean().to(device)
        loss = policy_loss + entropy_reg

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        ep_rewards.append(sum(rewards))
        ep_accuracy.append(float(correct))
        flag_rates.append(1.0 if flag_pos is not None else 0.0)

        if (ep + 1) % 100 == 0:
            recent_acc  = np.mean(ep_accuracy[-100:])
            recent_rew  = np.mean(ep_rewards[-100:])
            recent_flag = np.mean(flag_rates[-100:])
            flag_info   = (
                f"flagged at {flag_pos}/{total_tokens} "
                f"({100*flag_pos/total_tokens:.0f}%)"
                if flag_pos else "did not flag"
            )
            print(f"  Ep {ep+1:5d} | acc={recent_acc:.3f} | "
                  f"reward={recent_rew:.3f} | "
                  f"flag_rate={recent_flag:.2f} | "
                  f"{flag_info}")

            # early stop if converged
            if ep > 500 and recent_acc > 0.75:
                print(f"  Converged at episode {ep+1} ✓")
                break

            if recent_acc > best_acc:
                best_acc = recent_acc
                torch.save(policy.state_dict(),
                           f'{STEP2_DIR}/rl_policy_best.pt')

    print(f"\n  Best accuracy : {best_acc:.4f}")
    print(f"  Final flag rate: {np.mean(flag_rates[-100:]):.2f} "
          f"(should be 0.4–0.7, not 1.0)")
    return policy, ep_rewards, ep_accuracy

print("RL framework (fixed) loaded ✓")
print(f"Will train at L* = {tqa_layer} — episodes=2000, lr=5e-4")

In [ ]:
# ── CELL 10 — EXPERIMENT F: RL policy (offline, fixed) ───────────
# Instead of online token-by-token REINFORCE (which collapses),
# we pre-extract hidden states at multiple cutoffs and train
# the policy offline to distinguish factual vs hallucinated.
# This is equivalent to RL with a shaped reward but much more stable.

class HallucinationPolicy(nn.Module):
    """Same architecture, different training."""
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_policy_offline(train_data, target_layer,
                         cutoffs=None, epochs=20,
                         batch_size=64, lr=1e-3,
                         device='cuda'):
    """
    Offline training on pre-extracted hidden states.

    For each sample at each cutoff, extract hidden state
    and train policy to output:
      - high probability for hallucinated samples
      - low probability for factual samples

    Early detection bonus: weight loss inversely by cutoff
    so the policy is rewarded more for correct decisions
    made with fewer tokens.
    """
    if cutoffs is None:
        cutoffs = [0.10, 0.20, 0.30, 0.50, 1.00]

    print(f"  Pre-extracting hidden states at cutoffs: "
          f"{[int(c*100) for c in cutoffs]}%...")

    all_hiddens = []
    all_labels  = []
    all_weights = []   # early detection weight per sample

    for cutoff in cutoffs:
        h, l = extract_hidden_states(train_data, target_layer, cutoff)
        # early weight: more reward for decisions at fewer tokens
        early_weight = 1.0 + 1.0 * (1.0 - cutoff)
        all_hiddens.append(h)
        all_labels.append(l.float())
        all_weights.append(
            torch.full((len(h),), early_weight)
        )
        print(f"    Cutoff {int(cutoff*100):3d}%: "
              f"{len(h)} samples, weight={early_weight:.1f}")

    h_all = torch.cat(all_hiddens, dim=0)
    l_all = torch.cat(all_labels,  dim=0)
    w_all = torch.cat(all_weights, dim=0)

    hidden_dim = h_all.shape[1]
    policy     = HallucinationPolicy(hidden_dim, hidden_dim=128).to(device)
    optimizer  = torch.optim.Adam(policy.parameters(), lr=lr)
    criterion  = nn.BCELoss(reduction='none')

    print(f"\n  Training offline policy...")
    print(f"  Total samples: {len(h_all)} "
          f"({len(cutoffs)} cutoffs × {len(train_data)} samples)")

    # balance classes
    pos_idx = (l_all == 0).nonzero().squeeze()   # hallucinated
    neg_idx = (l_all == 1).nonzero().squeeze()   # factual
    min_n   = min(len(pos_idx), len(neg_idx))
    bal_idx = torch.cat([pos_idx[:min_n], neg_idx[:min_n]])

    h_bal = h_all[bal_idx]
    l_bal = l_all[bal_idx]   # 0=factual(don't flag), 1=hallu(flag)
    # flip labels: policy outputs P(hallucinated)
    # label=0 → factual → don't flag → target=0
    # label=1 → hallucinated → flag → target=1
    # HaluEval: factuality_score>0.5 → factual → label=1
    # so hallucinated label in BCE = 1 - l_bal
    target_bal = 1.0 - l_bal   # 1=hallucinated, 0=factual
    w_bal      = w_all[bal_idx]

    dataset    = torch.utils.data.TensorDataset(
        h_bal, target_bal, w_bal)
    dataloader = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=True)

    best_loss = float('inf')
    for epoch in range(epochs):
        epoch_loss = 0
        policy.train()
        for h_batch, t_batch, w_batch in dataloader:
            h_batch = h_batch.to(device)
            t_batch = t_batch.to(device)
            w_batch = w_batch.to(device)

            optimizer.zero_grad()
            probs = policy(h_batch)
            loss  = (criterion(probs, t_batch) * w_batch).mean()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs} | "
                  f"loss={avg_loss:.4f}")
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(policy.state_dict(),
                       f'{STEP2_DIR}/rl_policy_offline_best.pt')

    policy.eval()
    print(f"\n  Best loss: {best_loss:.4f}")
    print(f"  Saved ✓")
    return policy

print("Offline RL policy framework loaded ✓")


# ── CELL 11 — EXPERIMENT F: train offline policy ─────────────────

print("\n" + "="*60)
print("  EXPERIMENT F — Offline adaptive detection policy")
print("  Trained on pre-extracted hidden states at 5 cutoffs")
print("="*60)

tqa_train_rl = datasets['TriviaQA'][:int(
    0.8 * len(datasets['TriviaQA']))]

print(f"\nTarget L* = {tqa_layer} | "
      f"Train samples = {len(tqa_train_rl)}")

rl_policy = train_policy_offline(
    tqa_train_rl,
    target_layer=tqa_layer,
    cutoffs=[0.10, 0.20, 0.30, 0.50, 1.00],
    epochs=20,
    batch_size=64,
    lr=1e-3,
    device=DEVICE
)

In [ ]:
# ── Check separability at each cutoff independently ───────────────
print("Separability check at each cutoff (layer 13):")
print(f"{'Cutoff':>8} {'AUROC':>8} {'Separable?':>12}")
print("-" * 32)

tqa_eval_check = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]

for cutoff in [0.10, 0.20, 0.30, 0.50, 1.00]:
    h, l  = extract_hidden_states(tqa_eval_check, tqa_layer, cutoff)
    auroc = quick_auroc(h, l, cv=3)
    sep   = "✓ yes" if auroc > 0.65 else ("~ weak" if auroc > 0.55 else "✗ no")
    print(f"{int(cutoff*100):>7}%   {auroc:.4f}   {sep}")

In [ ]:
# ── Retrain offline policy on good cutoffs only ───────────────────
# Only use 20% and 100% — the only two with learnable signal
# Drop 10%, 30%, 50% — they're noise that hurts training

print("Retraining on separable cutoffs only: [20%, 100%]")
print("Dropping: 10% (AUROC=0.47), 30% (0.62), 50% (0.47)\n")

tqa_train_rl = datasets['TriviaQA'][:int(
    0.8 * len(datasets['TriviaQA']))]

rl_policy = train_policy_offline(
    tqa_train_rl,
    target_layer=tqa_layer,
    cutoffs=[0.20, 1.00],   # only separable cutoffs
    epochs=30,              # more epochs since less data
    batch_size=32,          # smaller batch for small dataset
    lr=5e-4,                # lower lr
    device=DEVICE
)

In [ ]:
# ── Evaluate retrained policy ─────────────────────────────────────
tqa_eval_rl = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]

print("Evaluating at 20% cutoff (the earliest reliable point):")
rl_results_20 = evaluate_rl_policy(
    rl_policy, tqa_eval_rl, tqa_layer, device=DEVICE)

print(f"\n  AUROC       : {rl_results_20['auroc']:.4f}")
print(f"  Accuracy    : {rl_results_20['accuracy']:.4f}")
print(f"  Flag rate   : {rl_results_20['flag_rate']*100:.1f}%")
print(f"  Avg flag at : {rl_results_20['avg_flag_pct']*100:.1f}% of tokens")

# compare against raw probe at same cutoffs
print(f"\n  Raw probe @ 20%  : "
      f"{early_results['TriviaQA'][0.20]['probe_auroc']:.4f}")
print(f"  Raw probe @ 100% : "
      f"{early_results['TriviaQA'][1.00]['probe_auroc']:.4f}")
print(f"  Policy AUROC     : {rl_results_20['auroc']:.4f}")

In [ ]:
# ── Check what probabilities the policy actually outputs ──────────
print("Policy output distribution on eval set:")

tqa_eval_rl = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]

factual_probs = []
hallu_probs   = []

for item in tqa_eval_rl[:80]:
    full_text = item['prompt'] + " " + item['response']
    true_label = 1 if item['factuality_score'] > 0.5 else 0
    inputs = tokenizer(full_text, return_tensors='pt',
                       truncation=True, max_length=256).to(DEVICE)
    total_tokens = inputs['input_ids'].shape[1]
    cutoff = max(4, int(0.2 * total_tokens))
    trunc  = {k: v[:, :cutoff] for k, v in inputs.items()}

    with torch.no_grad():
        out  = model(**trunc, output_hidden_states=True)
        h    = out.hidden_states[tqa_layer][0, -1, :].float()
        prob = rl_policy(h.unsqueeze(0).to(DEVICE)).item()

    if true_label == 1:
        factual_probs.append(prob)
    else:
        hallu_probs.append(prob)

print(f"  Factual samples   — mean prob: {np.mean(factual_probs):.4f} "
      f"min: {np.min(factual_probs):.4f} "
      f"max: {np.max(factual_probs):.4f}")
print(f"  Hallucinated samples — mean prob: {np.mean(hallu_probs):.4f} "
      f"min: {np.min(hallu_probs):.4f} "
      f"max: {np.max(hallu_probs):.4f}")
print(f"\n  Separation: {np.mean(hallu_probs) - np.mean(factual_probs):+.4f}")
print(f"  (positive = policy gives higher prob to hallucinated = correct direction)")

In [ ]:
# ── CELL 12 — Final evaluation: linear probe as early detector ────

print("\n" + "="*60)
print("  FINAL EVALUATION — Linear probe early detection")
print("="*60)

tqa_eval = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]

print(f"\n{'Cutoff':>8} {'AUROC':>8} {'±Std':>8} {'Interpretation'}")
print("-" * 55)

cutoff_results = {}
for cutoff in [0.10, 0.20, 0.30, 0.50, 1.00]:
    h, l       = extract_hidden_states(tqa_eval, tqa_layer, cutoff)
    mean, std  = robust_auroc(h, l, cv=10)
    cutoff_results[cutoff] = (mean, std)

    if   mean > 0.75: interp = "strong detection"
    elif mean > 0.65: interp = "reliable detection"
    elif mean > 0.55: interp = "weak signal"
    else:             interp = "chance — no signal"

    print(f"{int(cutoff*100):>7}%   {mean:.4f}   ±{std:.4f}   {interp}")

print(f"\n── Key result ────────────────────────────────────────")
best_early = max(
    [c for c in cutoff_results if c < 1.0],
    key=lambda c: cutoff_results[c][0]
)
best_early_auroc = cutoff_results[best_early][0]
full_auroc       = cutoff_results[1.00][0]

print(f"  Earliest reliable detection : "
      f"{int(best_early*100)}% tokens | "
      f"AUROC = {best_early_auroc:.4f}")
print(f"  Full sequence detection     : "
      f"100% tokens | "
      f"AUROC = {full_auroc:.4f}")
print(f"  Cost of early detection     : "
      f"{full_auroc - best_early_auroc:+.4f} AUROC")
print(f"  Token savings               : "
      f"{100 - int(best_early*100)}% fewer tokens")

# ── CELL 13 — Cross-dataset early detection ───────────────────────

print("\n" + "="*60)
print("  CROSS-DATASET EARLY DETECTION RESULTS")
print("="*60)

print(f"\n{'Dataset':<14} {'L*':>4} "
      f"{'20% AUROC':>10} {'100% AUROC':>12} {'Cost':>8}")
print("-" * 52)

cross_early = {}
for ds_name, ds_data in datasets.items():
    ds_eval  = ds_data[int(0.8*len(ds_data)):]
    ds_layer = full_results[ds_name]['best_layer']

    h20,  l20  = extract_hidden_states(ds_eval, ds_layer, 0.20)
    h100, l100 = extract_hidden_states(ds_eval, ds_layer, 1.00)

    auroc_20,  _ = robust_auroc(h20,  l20,  cv=5)
    auroc_100, _ = robust_auroc(h100, l100, cv=5)
    cost         = auroc_100 - auroc_20

    cross_early[ds_name] = {
        'layer':     ds_layer,
        'auroc_20':  auroc_20,
        'auroc_100': auroc_100,
        'cost':      cost
    }
    print(f"{ds_name:<14} {ds_layer:>4}  "
          f"{auroc_20:>10.4f}  "
          f"{auroc_100:>12.4f}  "
          f"{cost:>+8.4f}")

# paper claim
print(f"\n── Paper claim ───────────────────────────────────────")
mean_20  = np.mean([v['auroc_20']  for v in cross_early.values()])
mean_100 = np.mean([v['auroc_100'] for v in cross_early.values()])
print(f"  Mean AUROC @ 20% tokens  : {mean_20:.4f}")
print(f"  Mean AUROC @ 100% tokens : {mean_100:.4f}")
print(f"  Mean cost of early det.  : {mean_100-mean_20:+.4f}")
print(f"\n  → A linear probe at the auto-selected critical layer")
print(f"    achieves mean AUROC {mean_20:.3f} using only 20% of")
print(f"    generated tokens across 4 diverse QA datasets.")

In [ ]:
# ── CELL 10 — RL layered on linear probe ─────────────────────────
# State  : probe's P(hallucinated) score at each token cutoff
# Action : continuous adjustment to the probe threshold
# Reward : +1 if probe score correctly reflects true label
#          scaled by how confidently it reflects it
#          factual drift → score near 0 → positive reward
#          hallu drift   → score near 1 → positive reward

import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import roc_auc_score

def get_probe_scores(probe, data, target_layer,
                     cutoff_pct=1.0):
    """
    Run linear probe on hidden states at given cutoff.
    Returns probe scores P(hallucinated) and true labels.

    This is the state the RL policy sees.
    """
    h, l  = extract_hidden_states(data, target_layer, cutoff_pct)
    X     = h.numpy()
    # probe.predict_proba returns [P(factual), P(hallu)]
    proba = probe.predict_proba(X)[:, 1]   # P(hallucinated)
    return proba, l.numpy()

def compute_probe_reward(probe_score, true_label, cutoff_pct):
    """
    Reward based on how well probe score reflects true label.

    Factual sample (true_label=1):
      probe_score near 0 → correct → positive reward
      probe_score near 1 → wrong   → negative reward
      reward = (1 - probe_score) * early_weight

    Hallucinated sample (true_label=0):
      probe_score near 1 → correct → positive reward
      probe_score near 0 → wrong   → negative reward
      reward = probe_score * early_weight

    early_weight: higher reward for correct decisions
    made with fewer tokens
    """
    early_weight = 1.0 + (1.0 - cutoff_pct)

    if true_label == 1:      # factual
        # reward = how far probe score is from 1 (hallucination)
        # perfect score = 0.0 → reward = 1.0 * weight
        # wrong score   = 1.0 → reward = 0.0 * weight
        reward = (1.0 - probe_score) * early_weight
    else:                    # hallucinated
        # reward = how close probe score is to 1
        # perfect score = 1.0 → reward = 1.0 * weight
        # wrong score   = 0.0 → reward = 0.0 * weight
        reward = probe_score * early_weight

    # center reward around 0 so policy has clear signal
    # 0.5 = chance level = no reward, above = good, below = bad
    return reward - 0.5 * early_weight

def train_rl_on_probe(probe, train_data, target_layer,
                      cutoffs=None, epochs=50,
                      lr=1e-3, device='cuda'):
    """
    Train RL policy layered on top of linear probe.

    The policy learns a calibration — it takes the probe's
    score at each cutoff and learns whether that score is
    reliable or needs adjustment.

    State  : probe score P(hallu) at cutoff t
    Action : adjusted threshold for this cutoff
    Reward : how well the adjusted threshold separates
             factual from hallucinated

    Concretely: we learn a per-cutoff temperature scaling
    that makes the probe more reliable at early cutoffs.
    """
    if cutoffs is None:
        cutoffs = [0.20, 0.30, 0.50, 1.00]

    print("Pre-computing probe scores at all cutoffs...")

    # collect probe scores at each cutoff
    cutoff_data = {}
    for cutoff in cutoffs:
        scores, labels = get_probe_scores(
            probe, train_data, target_layer, cutoff)
        cutoff_data[cutoff] = {
            'scores': torch.tensor(scores, dtype=torch.float32),
            'labels': torch.tensor(labels, dtype=torch.float32)
        }
        auroc = roc_auc_score(1 - labels, scores)
        print(f"  Cutoff {int(cutoff*100):3d}%: "
              f"{len(scores)} samples | "
              f"probe AUROC = {auroc:.4f}")

    # learnable temperature per cutoff
    # temperature scales the probe score before thresholding
    # temp > 1 → sharpen (more confident)
    # temp < 1 → soften (less confident)
    temperatures = nn.ParameterDict({
        str(c): nn.Parameter(torch.ones(1))
        for c in cutoffs
    }).to(device)

    # learnable bias per cutoff
    # shifts the decision threshold
    biases = nn.ParameterDict({
        str(c): nn.Parameter(torch.zeros(1))
        for c in cutoffs
    }).to(device)

    optimizer = torch.optim.Adam(
        list(temperatures.values()) + list(biases.values()),
        lr=lr
    )

    total_params = sum(p.numel() for p in temperatures.values()) + \
                   sum(p.numel() for p in biases.values())
    print(f"\nRL policy parameters: {total_params} "
          f"(2 per cutoff — temperature + bias)")
    print(f"Cutoffs: {[int(c*100) for c in cutoffs]}%")
    print(f"\nTraining...")

    best_mean_auroc = 0
    history = []

    for epoch in range(epochs):
        epoch_rewards = {c: [] for c in cutoffs}
        total_loss    = torch.tensor(0.0, requires_grad=True)

        for cutoff in cutoffs:
            scores = cutoff_data[cutoff]['scores'].to(device)
            labels = cutoff_data[cutoff]['labels'].to(device)

            temp = temperatures[str(cutoff)]
            bias = biases[str(cutoff)]

            # apply temperature scaling and bias to probe scores
            # this is the RL policy's action
            calibrated = torch.sigmoid(
                temp * torch.logit(scores.clamp(0.01, 0.99)) + bias
            )

            # compute reward for each sample
            rewards = torch.where(
                labels == 1,
                # factual: reward for low hallucination score
                (1.0 - calibrated) * (1.0 + (1.0 - cutoff)),
                # hallucinated: reward for high hallucination score
                calibrated * (1.0 + (1.0 - cutoff))
            )
            # center rewards
            rewards = rewards - 0.5 * (1.0 + (1.0 - cutoff))

            # loss = negative reward (maximize reward)
            cutoff_loss = -rewards.mean()
            total_loss  = total_loss + cutoff_loss

            epoch_rewards[cutoff].append(rewards.mean().item())

        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        # log every 10 epochs
        if (epoch + 1) % 10 == 0:
            # evaluate calibrated AUROC at each cutoff
            aurocs = {}
            with torch.no_grad():
                for cutoff in cutoffs:
                    scores = cutoff_data[cutoff]['scores'].to(device)
                    labels = cutoff_data[cutoff]['labels'].numpy()
                    temp   = temperatures[str(cutoff)]
                    bias   = biases[str(cutoff)]
                    cal    = torch.sigmoid(
                        temp * torch.logit(
                            scores.clamp(0.01, 0.99)) + bias
                    ).cpu().numpy()
                    aurocs[cutoff] = roc_auc_score(1 - labels, cal)

            mean_auroc = np.mean(list(aurocs.values()))
            auroc_str  = " | ".join(
                f"{int(c*100)}%:{aurocs[c]:.3f}" for c in cutoffs)
            print(f"  Epoch {epoch+1:3d} | "
                  f"mean_auroc={mean_auroc:.4f} | "
                  f"{auroc_str}")

            if mean_auroc > best_mean_auroc:
                best_mean_auroc = mean_auroc
                best_temps = {
                    k: v.item()
                    for k, v in temperatures.items()
                }
                best_biases = {
                    k: v.item()
                    for k, v in biases.items()
                }

            history.append({
                'epoch':       epoch + 1,
                'mean_auroc':  mean_auroc,
                'aurocs':      aurocs.copy()
            })

    print(f"\nBest mean AUROC: {best_mean_auroc:.4f}")
    print(f"\nLearned temperatures (probe sharpening per cutoff):")
    for c in cutoffs:
        t = temperatures[str(c)].item()
        b = biases[str(c)].item()
        effect = "sharper" if t > 1 else "softer"
        print(f"  {int(c*100):3d}% → temp={t:.3f} ({effect}) "
              f"bias={b:+.3f}")

    return temperatures, biases, history, best_mean_auroc

print("RL-on-probe framework loaded ✓")
print(f"Target L* = {tqa_layer}")
print(f"\nWhat this does:")
print(f"  1. Takes probe P(hallu) score as state")
print(f"  2. Learns temperature + bias per cutoff")
print(f"  3. Rewards factual scores near 0 (correct drift)")
print(f"  4. Rewards hallu scores near 1 (correct drift)")
print(f"  5. Produces calibrated probe that works at early cutoffs")


# ── CELL 11 — Train RL on probe ───────────────────────────────────

print("\n" + "="*60)
print("  EXPERIMENT F — RL calibration layered on linear probe")
print("="*60)

# get the trained probe from Experiment A
tqa_probe      = full_results['TriviaQA']['probe']
tqa_train_data = datasets['TriviaQA'][:int(
    0.8 * len(datasets['TriviaQA']))]

print(f"\nProbe trained at L* = {tqa_layer}")
print(f"RL will calibrate probe scores at each cutoff\n")

temperatures, biases, history, best_auroc = train_rl_on_probe(
    tqa_probe,
    tqa_train_data,
    target_layer=tqa_layer,
    cutoffs=[0.20, 0.30, 0.50, 1.00],
    epochs=50,
    lr=1e-3,
    device=DEVICE
)


# ── CELL 12 — Evaluate calibrated probe ──────────────────────────

print("\n" + "="*60)
print("  EXPERIMENT F — Evaluation: raw probe vs RL-calibrated")
print("="*60)

tqa_eval_data = datasets['TriviaQA'][int(
    0.8 * len(datasets['TriviaQA'])):]

print(f"\n{'Cutoff':>8} {'Raw probe':>12} "
      f"{'RL calibrated':>14} {'Delta':>8}")
print("-" * 48)

eval_results = {}
for cutoff in [0.20, 0.30, 0.50, 1.00]:
    # raw probe score
    raw_scores, labels = get_probe_scores(
        tqa_probe, tqa_eval_data, tqa_layer, cutoff)
    raw_auroc = roc_auc_score(1 - labels, raw_scores)

    # RL-calibrated score
    raw_t = torch.tensor(raw_scores, dtype=torch.float32)
    temp  = temperatures[str(cutoff)]
    bias  = biases[str(cutoff)]
    with torch.no_grad():
        cal_scores = torch.sigmoid(
            temp * torch.logit(raw_t.clamp(0.01, 0.99)) + bias
        ).numpy()

    cal_auroc = roc_auc_score(1 - labels, cal_scores)
    delta     = cal_auroc - raw_auroc

    eval_results[cutoff] = {
        'raw_auroc': raw_auroc,
        'cal_auroc': cal_auroc,
        'delta':     delta
    }
    print(f"{int(cutoff*100):>7}%  {raw_auroc:>12.4f}  "
          f"{cal_auroc:>14.4f}  {delta:>+8.4f}")

print(f"\n── What RL learned ───────────────────────────────────")
for cutoff in [0.20, 0.30, 0.50, 1.00]:
    t = temperatures[str(cutoff)].item()
    b = biases[str(cutoff)].item()
    r = eval_results[cutoff]
    print(f"  {int(cutoff*100):3d}% tokens | "
          f"temp={t:.3f} bias={b:+.3f} | "
          f"AUROC {r['raw_auroc']:.4f} → {r['cal_auroc']:.4f} "
          f"({r['delta']:+.4f})")


# ── CELL 13 — Cross-dataset calibrated probe ─────────────────────

print(f"\n── Cross-dataset: RL-calibrated probe ────────────────")
print(f"  Calibration learned on TriviaQA")
print(f"  Testing transfer to other datasets\n")

print(f"  {'Dataset':<14} {'Raw 20%':>9} "
      f"{'Cal 20%':>9} {'Delta':>8}")
print(f"  {'-'*44}")

for ds_name, ds_data in datasets.items():
    ds_eval = ds_data[int(0.8 * len(ds_data)):]
    ds_layer = full_results[ds_name]['best_layer']
    ds_probe = full_results[ds_name]['probe']

    raw_scores, labels = get_probe_scores(
        ds_probe, ds_eval, ds_layer, 0.20)
    raw_auroc = roc_auc_score(1 - labels, raw_scores)

    raw_t = torch.tensor(raw_scores, dtype=torch.float32)
    temp  = temperatures[str(0.20)]
    bias  = biases[str(0.20)]
    with torch.no_grad():
        cal_scores = torch.sigmoid(
            temp * torch.logit(raw_t.clamp(0.01, 0.99)) + bias
        ).numpy()
    cal_auroc = roc_auc_score(1 - labels, cal_scores)

    print(f"  {ds_name:<14} {raw_auroc:>9.4f} "
          f"{cal_auroc:>9.4f} {cal_auroc-raw_auroc:>+8.4f}")

print(f"\n── Paper claim ───────────────────────────────────────")
mean_delta = np.mean([
    eval_results[c]['delta'] for c in eval_results
])
print(f"  RL calibration mean delta: {mean_delta:+.4f}")
print(f"  Learned temperature per cutoff shows which")
print(f"  token positions need sharpening vs softening")
print(f"  to make the probe reliable at early detection")